## 1. Verify the current Colab environment

Do not reinstall or downgrade NumPy/XGBoost if the current runtime already works.


In [1]:
import sys, numpy as np, pandas as pd, sklearn, xgboost as xgb, torch
import transformers, datasets, peft, accelerate, bitsandbytes

print("========== ENVIRONMENT ==========")
print("Python:", sys.version.split()[0])
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgb.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)


KeyboardInterrupt: 

If imports fail because the current Colab runtime is missing a package, install only the missing package and restart once. Otherwise continue without changing the environment.


In [2]:
# Optional: only run if the previous verification shows a missing package.
# Example:
!pip install -q transformers datasets peft accelerate bitsandbytes
#
# Do NOT downgrade NumPy or XGBoost just because a different notebook used older versions.
print("Environment setup is intentionally minimal.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.7 MB/s eta 0:00:00
Environment setup is intentionally minimal.


## 2. Verify the environment after restart

In [3]:
import torch
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
import transformers
import peft

print("========== ENVIRONMENT ==========")
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("XGBoost:", xgb.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("\nEnvironment ready.")

========== ENVIRONMENT ==========
GPU available: True
GPU: Tesla T4
NumPy: 2.1.3
Pandas: 2.2.3
Scikit-learn: 1.6.1
XGBoost: 3.4.1
Transformers: 5.16.1
PEFT: 0.20.0

Environment ready.


## 3. Upload the three real PharmGKB annotation files

In [4]:
from google.colab import files

print("Upload:")
print("  var_fa_ann.tsv")
print("  var_drug_ann.tsv")
print("  var_pheno_ann.tsv")

uploaded = files.upload()

for f in uploaded:
    print("✓", f)

Upload:
  var_fa_ann.tsv
  var_drug_ann.tsv
  var_pheno_ann.tsv


Saving var_fa_ann.tsv to var_fa_ann.tsv
Saving var_drug_ann.tsv to var_drug_ann.tsv
Saving var_pheno_ann.tsv to var_pheno_ann.tsv
✓ var_fa_ann.tsv
✓ var_drug_ann.tsv
✓ var_pheno_ann.tsv


## 4. Load and combine PharmGKB

In [5]:
import pandas as pd

fa = pd.read_csv("var_fa_ann.tsv", sep="\t", low_memory=False)
drug = pd.read_csv("var_drug_ann.tsv", sep="\t", low_memory=False)
pheno = pd.read_csv("var_pheno_ann.tsv", sep="\t", low_memory=False)

def prepare(df):
    cols = [
        "Variant Annotation ID", "Variant/Haplotypes", "Gene", "Drug(s)",
        "Sentence", "Phenotype Category", "Significance",
        "Is/Is Not associated", "Direction of effect", "Alleles",
        "Metabolizer types"
    ]
    out = df[[c for c in cols if c in df.columns]].copy()
    out = out.dropna(subset=["Sentence"])
    out = out[out["Sentence"].astype(str).str.strip() != ""]
    return out

annotations = pd.concat(
    [prepare(fa), prepare(drug), prepare(pheno)],
    ignore_index=True
)

annotations = annotations.drop_duplicates(
    subset=["Variant Annotation ID", "Sentence"]
).reset_index(drop=True)

print("FA:", fa.shape)
print("Drug:", drug.shape)
print("Pheno:", pheno.shape)
print("Combined unique annotations:", annotations.shape)
display(annotations[[
    "Gene", "Drug(s)", "Sentence", "Phenotype Category",
    "Significance", "Direction of effect"
]].head(10))

FA: (2160, 23)
Drug: (13020, 22)
Pheno: (14534, 25)
Combined unique annotations: (29714, 11)


,Gene,Drug(s),Sentence,Phenotype Category,Significance,Direction of effect
0,CYP2C19,normeperidine,CYP2C19 *17/*17 is associated with increased f...,NaN,not stated,increased
1,CYP2B6,bupropion,CYP2B6 *1/*1 is associated with increased acti...,Efficacy,yes,increased
2,CYP2C9,warfarin,CYP2C9 *2 is associated with decreased affinit...,Metabolism/PK,not stated,decreased
3,CYP2C9,warfarin,CYP2C9 *3 is associated with decreased affinit...,Metabolism/PK,not stated,decreased
4,SLC22A1,ranitidine,Allele del is associated with decreased transp...,Metabolism/PK,yes,decreased
5,CYP2C19,voriconazole,CYP2C19 *32 is associated with decreased clear...,Metabolism/PK,yes,decreased
6,CYP2C19,voriconazole,CYP2C19 *31 is associated with decreased clear...,Metabolism/PK,yes,decreased
7,CYP2C19,voriconazole,CYP2C19 *30 is associated with decreased clear...,Metabolism/PK,yes,decreased
8,CYP2C19,"mephenytoin, omeprazole",Allele A is associated with decreased activity...,Metabolism/PK,yes,decreased
9,CYP2C19,voriconazole,CYP2C19 *29 is associated with decreased clear...,Metabolism/PK,yes,decreased


## 5. Upload pair-aware split and create train/validation/test

In [6]:
from google.colab import files

print("Upload: pair_aware_split.csv")
files.upload()

split_df = pd.read_csv("pair_aware_split.csv")

split_df["gene_norm"] = split_df["gene_norm"].astype(str).str.strip().str.upper()
split_df["drug_norm"] = split_df["drug_norm"].astype(str).str.strip().str.lower()

annotations["gene_norm"] = annotations["Gene"].astype(str).str.strip().str.upper()
annotations["drug_norm"] = annotations["Drug(s)"].astype(str).str.strip().str.lower()

annotations_split = annotations.merge(
    split_df[["gene_norm", "drug_norm", "gold_relation", "split"]],
    on=["gene_norm", "drug_norm"],
    how="inner"
)

print("Matched annotation examples:", len(annotations_split))
print(annotations_split["split"].value_counts())

Upload: pair_aware_split.csv


Saving pair_aware_split.csv to pair_aware_split.csv
Matched annotation examples: 22078
split
train         14636
test           4730
validation     2712
Name: count, dtype: int64


## 6. Build Qwen instruction datasets

In [7]:
import json
from datasets import Dataset

FIELDS = [
    "gene", "drug", "variant_or_haplotype", "phenotype_category",
    "significance", "association", "direction", "alleles",
    "metabolizer_types"
]

def clean_value(v):
    if pd.isna(v):
        return None
    v = str(v).strip()
    return None if v.lower() in {"nan", "none", ""} else v

def make_hf_dataset(data):
    records = []
    for _, row in data.iterrows():
        target = {
            "gene": clean_value(row["Gene"]),
            "drug": clean_value(row["Drug(s)"]),
            "variant_or_haplotype": clean_value(row["Variant/Haplotypes"]),
            "phenotype_category": clean_value(row["Phenotype Category"]),
            "significance": clean_value(row["Significance"]),
            "association": clean_value(row["Is/Is Not associated"]),
            "direction": clean_value(row["Direction of effect"]),
            "alleles": clean_value(row["Alleles"]),
            "metabolizer_types": clean_value(row["Metabolizer types"])
        }
        # Keep a fixed schema. Missing source information becomes null.
        records.append({
            "sentence": clean_value(row["Sentence"]),
            "target": json.dumps(target, ensure_ascii=False)
        })
    return Dataset.from_list(records)

train_data = annotations_split[annotations_split["split"] == "train"].copy()
val_data = annotations_split[annotations_split["split"] == "validation"].copy()
test_data = annotations_split[annotations_split["split"] == "test"].copy()

train_dataset = make_hf_dataset(train_data)
val_dataset = make_hf_dataset(val_data)
test_dataset = make_hf_dataset(test_data)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))
print(json.dumps(train_dataset[0], indent=2, ensure_ascii=False))


Train: 14636
Validation: 2712
Test: 4730
{
  "sentence": "CYP2C19 *17/*17 is associated with increased formation of normeperidine as compared to CYP2C19 *1/*1 + *1/*17.",
  "target": "{\"gene\": \"CYP2C19\", \"drug\": \"normeperidine\", \"variant_or_haplotype\": \"CYP2C19*1, CYP2C19*17\", \"phenotype_category\": null, \"significance\": \"not stated\", \"association\": \"Associated with\", \"direction\": \"increased\", \"alleles\": \"*17/*17\", \"metabolizer_types\": null}"
}


## 7. Load Qwen 2.5 3B in 4-bit

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Qwen loaded successfully:", MODEL_NAME)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen loaded successfully: Qwen/Qwen2.5-3B-Instruct


## 8. Add LoRA

In [9]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 7,372,800 || all params: 3,093,311,488 || trainable%: 0.2383


## 9. Format and tokenize

In [10]:
SYSTEM_PROMPT = (
    "You are a pharmacogenomics information extraction model. "
    "Extract only the structured evidence explicitly described in the "
    "PharmGKB sentence. Do not infer facts that are not stated. "
    "Return only valid JSON using the requested fields."
)

def build_prompt_and_target(sentence, target):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": sentence}
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    target_text = target + tokenizer.eos_token
    return prompt_text, target_text

MAX_LENGTH = 384

def tokenize_example(example):
    prompt_text, target_text = build_prompt_and_target(
        example["sentence"], example["target"]
    )

    prompt_ids = tokenizer(
        prompt_text, add_special_tokens=False, truncation=False
    )["input_ids"]

    target_ids = tokenizer(
        target_text, add_special_tokens=False, truncation=False
    )["input_ids"]

    # Labels are -100 for the prompt and real token IDs only for the JSON.
    # Therefore the loss teaches extraction rather than prompt reproduction.
    if len(prompt_ids) + len(target_ids) > MAX_LENGTH:
        keep_prompt = max(0, MAX_LENGTH - len(target_ids))
        prompt_ids = prompt_ids[-keep_prompt:]
        target_ids = target_ids[:MAX_LENGTH - keep_prompt]

    input_ids = prompt_ids + target_ids
    labels = [-100] * len(prompt_ids) + target_ids
    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

train_tokenized = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
    desc="Tokenizing train"
)
val_tokenized = val_dataset.map(
    tokenize_example,
    remove_columns=val_dataset.column_names,
    desc="Tokenizing validation"
)

print("Train:", len(train_tokenized))
print("Validation:", len(val_tokenized))
print("Example length:", len(train_tokenized[0]["input_ids"]))
print("Supervised target tokens:",
      sum(x != -100 for x in train_tokenized[0]["labels"]))


Tokenizing train:   0%|          | 0/14636 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/2712 [00:00<?, ? examples/s]

Train: 14636
Validation: 2712
Example length: 191
Supervised target tokens: 95


## 10. Create the trainer

In [11]:
from transformers import TrainingArguments

FULL_OUTPUT_DIR = "./qwen_pharmgkb_full"

full_args = TrainingArguments(
    output_dir=FULL_OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    fp16=True,
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    logging_steps=25,
    gradient_checkpointing=True,
    report_to="none",
    seed=42
)

print("Full-training configuration ready.")


Full-training configuration ready.


## 11. Train Qwen

In [12]:
SMOKE_TRAIN_SIZE = 500
SMOKE_VAL_SIZE = 200

train_small = train_tokenized.select(
    range(min(SMOKE_TRAIN_SIZE, len(train_tokenized)))
)
val_small = val_tokenized.select(
    range(min(SMOKE_VAL_SIZE, len(val_tokenized)))
)

print("Smoke train:", len(train_small))
print("Smoke validation:", len(val_small))


Smoke train: 500
Smoke validation: 200


In [13]:
def qwen_collator(features):
    max_len = max(len(f["input_ids"]) for f in features)

    input_ids, attention_masks, labels = [], [], []

    for f in features:
        pad_len = max_len - len(f["input_ids"])
        input_ids.append(
            f["input_ids"] + [tokenizer.pad_token_id] * pad_len
        )
        attention_masks.append(
            f["attention_mask"] + [0] * pad_len
        )
        labels.append(
            f["labels"] + [-100] * pad_len
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_masks, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long)
    }

import torch.nn.functional as F
from transformers import Trainer

class QwenCausalLMTrainer(Trainer):
    def compute_loss(
        self, model, inputs, return_outputs=False, num_items_in_batch=None
    ):
        labels = inputs["labels"]

        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )

        logits = outputs.logits
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100
        )

        return (loss, outputs) if return_outputs else loss


In [15]:
from transformers import TrainingArguments

smoke_args = TrainingArguments(
    output_dir="./qwen_smoke_test",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="no",
    logging_steps=10,
    gradient_checkpointing=True,
    report_to="none",
    seed=42
)

trainer_small = QwenCausalLMTrainer(
    model=model,
    args=smoke_args,
    train_dataset=train_small,
    eval_dataset=val_small,
    data_collator=qwen_collator
)

print("Smoke-test trainer ready.")


Smoke-test trainer ready.


In [16]:
print("Starting smoke test...")
smoke_result = trainer_small.train()
smoke_eval = trainer_small.evaluate()

print("\n==============================")
print("SMOKE TEST COMPLETE")
print("==============================")
print("Training loss:", smoke_result.training_loss)
print("Validation loss:", smoke_eval["eval_loss"])
print("\nIf this completed successfully, proceed to FULL training.")


Starting smoke test...


Step,Training Loss,Validation Loss
50,0.585992,0.112211
63,0.454245,0.112959


Training Loss,Validation Loss,Step
0.454245,0.112959,63



SMOKE TEST COMPLETE
Training loss: 1.522232176765563
Validation loss: 0.11295867711305618

If this completed successfully, proceed to FULL training.


## FULL TRAINING

The smoke test above is only a pipeline check. The next code cell trains on the full training set and validation set. The test set remains untouched until the held-out evaluation section.


In [ ]:
# ==============================
# FULL QWEN TRAINING
# ==============================

torch.cuda.empty_cache()

trainer = QwenCausalLMTrainer(
    model=model,
    args=full_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=qwen_collator
)

print("Starting FULL Qwen training...")

full_result = trainer.train()
full_eval = trainer.evaluate()

print("\n==============================")
print("FULL TRAINING COMPLETE")
print("==============================")
print("Training loss:", full_result.training_loss)
print("Validation loss:", full_eval["eval_loss"])
print("Best checkpoint:", trainer.state.best_model_checkpoint)


Starting FULL Qwen training...


Step,Training Loss,Validation Loss


In [ ]:
import os

print("Current directory:", os.getcwd())

for path in [
    "./qwen_pharmgkb_final",
    "./qwen_pharmgkb_final_adapter",
    "./qwen_smoke",
]:
    print("\n", path)
    if os.path.exists(path):
        print(os.listdir(path)[:20])
    else:
        print("NOT FOUND")

In [ ]:
# Test the Qwen model after the smoke-test training

test_sentence = test_data.iloc[0]["Sentence"]

print("PHARMGKB INPUT")
print("=" * 80)
print(test_sentence)

print("\nQWEN OUTPUT")
print("=" * 80)

prediction = qwen_extract(test_sentence)

print(prediction)

## 12. Save the LoRA adapter

In [ ]:
FINAL_ADAPTER_DIR = "./qwen_pharmgkb_final_adapter"

model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

print("Saved actual final LoRA adapter:", FINAL_ADAPTER_DIR)


## 13. Test Qwen on one held-out sentence

In [ ]:
import re, json, torch

def parse_json_output(text):
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if match:
            try:
                return json.loads(match.group(0))
            except Exception:
                return None
    return None

def qwen_extract(sentence, max_new_tokens=220):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": sentence}
    ]

    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(generated, skip_special_tokens=True).strip()

    return raw, parse_json_output(raw)

for i in range(5):
    sentence = test_data.iloc[i]["Sentence"]
    raw, parsed = qwen_extract(sentence)

    print("=" * 100)
    print("TEST EXAMPLE", i + 1)
    print("\nSOURCE:\n", sentence)
    print("\nQWEN RAW:\n", raw)
    print("\nPARSED JSON:\n", parsed)


## 14. Held-out extraction evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

EVAL_FIELDS = [
    "gene", "drug", "variant_or_haplotype",
    "phenotype_category", "significance", "association",
    "direction", "alleles", "metabolizer_types"
]

def normalize_text(x):
    if x is None or pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x).strip().lower())

def gold_record(row):
    return {
        "gene": clean_value(row["Gene"]),
        "drug": clean_value(row["Drug(s)"]),
        "variant_or_haplotype": clean_value(row["Variant/Haplotypes"]),
        "phenotype_category": clean_value(row["Phenotype Category"]),
        "significance": clean_value(row["Significance"]),
        "association": clean_value(row["Is/Is Not associated"]),
        "direction": clean_value(row["Direction of effect"]),
        "alleles": clean_value(row["Alleles"]),
        "metabolizer_types": clean_value(row["Metabolizer types"])
    }

# Evaluate once and cache the outputs so repeated notebook runs do not
# accidentally produce inconsistent result tables.
results = []

for i, (_, row) in enumerate(test_data.iterrows()):
    raw, pred = qwen_extract(row["Sentence"])
    results.append({
        "row_index": i,
        "sentence": row["Sentence"],
        "gold_relation": row["gold_relation"],
        "gold": gold_record(row),
        "prediction_raw": raw,
        "prediction": pred
    })

    if (i + 1) % 250 == 0:
        print(f"Evaluated {i+1}/{len(test_data)}")

eval_rows = []

for r in results:
    pred = r["prediction"] if isinstance(r["prediction"], dict) else {}
    out = {
        "row_index": r["row_index"],
        "sentence": r["sentence"],
        "gold_relation": r["gold_relation"],
        "prediction_raw": r["prediction_raw"],
        "valid_json": isinstance(r["prediction"], dict)
    }

    for field in EVAL_FIELDS:
        out["gold_" + field] = r["gold"].get(field)
        out["pred_" + field] = pred.get(field)

    eval_rows.append(out)

eval_df = pd.DataFrame(eval_rows)

print("\nValid JSON rate:", f"{eval_df['valid_json'].mean():.4%}")

# Field-level exact-match metrics on non-null gold targets.
metric_rows = []
for field in EVAL_FIELDS:
    gold = eval_df["gold_" + field].apply(normalize_text)
    pred = eval_df["pred_" + field].apply(normalize_text)
    mask = gold != ""

    y_true = gold[mask]
    y_pred = pred[mask]

    tp = ((y_true == y_pred) & (y_true != "")).sum()
    fp = ((y_pred != "") & (y_true != y_pred)).sum()
    fn = ((y_true != "") & (y_true != y_pred)).sum()

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0

    metric_rows.append({
        "field": field,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "n_gold": len(y_true)
    })

field_metrics = pd.DataFrame(metric_rows)

eval_df["full_extraction_correct"] = True
for field in EVAL_FIELDS:
    eval_df["full_extraction_correct"] &= (
        eval_df["gold_" + field].apply(normalize_text)
        == eval_df["pred_" + field].apply(normalize_text)
    )

print("Full-record exact match:",
      f"{eval_df['full_extraction_correct'].mean():.4%}")

display(field_metrics.style.format({
    "precision": "{:.4f}",
    "recall": "{:.4f}",
    "f1": "{:.4f}"
}))

# Sentence-level positive/negative association confusion matrix.
# Ambiguous/not-stated cases are excluded from this binary matrix.
def association_class(x):
    x = normalize_text(x)
    if not x:
        return "unknown"
    if "not associated" in x or "not significantly associated" in x:
        return "negative"
    if "associated" in x:
        return "positive"
    return "unknown"

eval_df["gold_association_class"] = eval_df["gold_association"].apply(association_class)
eval_df["pred_association_class"] = eval_df["pred_association"].apply(association_class)

cm_df = eval_df[
    eval_df["gold_association_class"].isin(["negative", "positive"]) &
    eval_df["pred_association_class"].isin(["negative", "positive"])
].copy()

labels = ["negative", "positive"]
cm = confusion_matrix(
    cm_df["gold_association_class"],
    cm_df["pred_association_class"],
    labels=labels
)

print("\nASSOCIATION CONFUSION MATRIX")
print("                  Predicted")
print("                 Negative  Positive")
print(f"Actual Negative   {cm[0,0]:8d}  {cm[0,1]:8d}")
print(f"Actual Positive   {cm[1,0]:8d}  {cm[1,1]:8d}")

print("\nClassification report:")
print(classification_report(
    cm_df["gold_association_class"],
    cm_df["pred_association_class"],
    labels=labels,
    target_names=["Negative", "Positive"],
    digits=4,
    zero_division=0
))

# Real held-out errors.
errors = eval_df[~eval_df["full_extraction_correct"]].copy()
errors["wrong_fields"] = ""

for idx in errors.index:
    wrong = []
    for field in EVAL_FIELDS:
        if normalize_text(errors.loc[idx, "gold_" + field]) != normalize_text(
            errors.loc[idx, "pred_" + field]
        ):
            wrong.append(field)
    errors.loc[idx, "wrong_fields"] = ", ".join(wrong)

print("\nTotal records with at least one extraction error:", len(errors))

for n, (_, row) in enumerate(errors.head(10).iterrows(), start=1):
    print("=" * 110)
    print(f"ERROR EXAMPLE {n}")
    print("\nSOURCE:\n", row["sentence"])
    print("\nGOLD:\n", {f: row["gold_" + f] for f in EVAL_FIELDS})
    print("\nQWEN:\n", {f: row["pred_" + f] for f in EVAL_FIELDS})
    print("\nWRONG FIELDS:", row["wrong_fields"])

# Save everything needed for Section 5.
eval_df.to_csv("qwen_heldout_test_results.csv", index=False)
errors.to_csv("qwen_real_errors.csv", index=False)
field_metrics.to_csv("qwen_field_metrics.csv", index=False)

print("\nSaved:")
print("qwen_heldout_test_results.csv")
print("qwen_real_errors.csv")
print("qwen_field_metrics.csv")


## 15. XGBoost side

The final model is **XGBoost regression only**. It will combine real IWPC genotype features with Qwen-derived evidence features. A real observed therapeutic-dose table is required before fitting and reporting clinical dose-prediction metrics.

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("XGBoost:", xgb.__version__)

print("""
FINAL ARCHITECTURE

PharmGKB evidence
        |
        v
Qwen 2.5 3B + LoRA
        |
        v
structured evidence
        |
        v
numerical evidence features
        +
IWPC genotype features
        |
        v
XGBoost regression
        |
        v
predicted warfarin therapeutic dose

Primary dose metrics:
MAE, RMSE, R^2

Optional:
percentage within +/-10%, +/-20%, +/-30% of observed dose

No Random Forest.
No fabricated clinical dose results.

A real observed therapeutic-dose phenotype table is required
before fitting or reporting clinical dose-prediction performance.
""")
